In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Add the root project directory (football-analytics) to path
sys.path.append(os.path.abspath('..'))
from Python.kaggle_loader import KaggleToSQLPipeline
from datetime import datetime
from dotenv import load_dotenv

In [8]:
# Call from .env file

dotenv_path = os.path.join('..', '.env')
load_dotenv(dotenv_path)

True

In [ ]:
# Connect to database

DB_USER = "root"

# Initialize the pipeline
pipe = KaggleToSQLPipeline(db_name="football", user=os.getenv('DB_USER'), password=os.getenv('DB_PASS'), port=os.getenv('DB_PORT'))

# Download the data and store in our local directory

data_folder = pipe.download_data("kamrangayibov/football-data-european-top-5-leagues")

file_path = os.path.join(data_folder, r'kaggle_dataset/')

Validating database connection...
Connection successful!
--- Downloading kamrangayibov/football-data-european-top-5-leagues ---
Data saved to: d:\Project\Data Analytics\venv\football_db\Codes\football-analytics\notebooks\datasets\kamrangayibov\football-data-european-top-5-leagues\versions\1


In [10]:
# Create a list containing only files that end with '.csv'
csv_files = [f for f in os.listdir(file_path) if f.endswith('.csv')]

print(csv_files)

['coaches.csv', 'leagues.csv', 'matches.csv', 'players.csv', 'referees.csv', 'scores.csv', 'seasons.csv', 'stadiums.csv', 'standings.csv', 'teams.csv']


In [11]:
# Preview the dataset using pandas dataframe

csv_to_load = os.path.join(file_path, "coaches.csv")

coach_df = pd.read_csv(csv_to_load)
display(coach_df.head(10))

,coach_id,name,team_id,nationality
0,34,NaN,354,NaN
1,35,NaN,356,NaN
2,36,NaN,389,NaN
3,37,NaN,397,NaN
4,38,NaN,402,NaN
5,39,NaN,563,NaN
6,40,NaN,1044,NaN
7,101,Kersten Kuhl,1,Germany
8,102,Pellegrino Matarazzo,2,USA
9,103,Xabi Alonso,3,Spain


In [12]:
# Preview the dataset using pandas dataframe

csv_to_load = os.path.join(file_path, "teams.csv")

team_df = pd.read_csv(csv_to_load)
display(team_df.head(10))

,team_id,name,founded_year,stadium_id,league_id,coach_id,cresturl
0,77,Athletic Club,1898.0,61,3,61,https://crests.football-data.org/77.png
1,78,Club Atlético de Madrid,1903.0,62,3,62,https://crests.football-data.org/78.svg
2,79,CA Osasuna,1920.0,63,3,63,https://crests.football-data.org/79.svg
3,81,FC Barcelona,1899.0,64,3,64,https://crests.football-data.org/81.svg
4,82,Getafe CF,1946.0,65,3,65,https://crests.football-data.org/82.png
5,83,Granada CF,1931.0,66,3,66,https://crests.football-data.org/83.svg
6,86,Real Madrid CF,1902.0,67,3,67,https://crests.football-data.org/86.png
7,87,Rayo Vallecano de Madrid,1924.0,68,3,68,https://crests.football-data.org/87.svg
8,89,RCD Mallorca,1916.0,69,3,69,https://crests.football-data.org/89.png
9,90,Real Betis Balompié,1907.0,70,3,70,https://crests.football-data.org/90.png


In [13]:
# Scrape manager information from Wikipedia
from Python.scrape_pl_manager import WikiTableParser

parser = WikiTableParser('https://en.wikipedia.org/wiki/List_of_Premier_League_managers')
pl_coach = parser.parse_managers_table()
if pl_coach is not None:
    print(pl_coach.head(3))

               Name      Nat.     Club              From             Until  \
1     George Graham  Scotland  Arsenal       14 May 1986  21 February 1995   
2  Stewart Houston‡  Scotland  Arsenal  22 February 1995       8 June 1995   
3       Bruce Rioch  Scotland  Arsenal       8 June 1995    12 August 1996   

  Duration(days) Years inLeague Ref.  
1           3205      1992–1995  [2]  
2            106           1995  [3]  
3            431      1995–1996  [2]  


In [14]:
# Check any values from "From" column that has a reference string
pl_coach[pl_coach['From'].astype(str).str.contains('\[.*\]', regex=True)]

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.
57,Gary O'Neil,England,Bournemouth,30 August 2022[a],19 June 2023,293,2022–2023,[36]
67,Roberto De Zerbi,Italy,Brighton & Hove Albion,18 September 2022[b],19 May 2024,1308,2022–2024,
181,Scott Parker,England,Fulham,28 February 2019[c],28 June 2021,851,20192020–2021,
187,Iain Dowie,Northern Ireland,Hull City,17 March 2010[d],9 May 2010,53,2010,[92]
189,Mike Phelan,England,Hull City,22 July 2016[e],3 January 2017,165,2016–2017,
217,Craig Shakespeare,England,Leicester City,23 February 2017[f],17 October 2017,236,2017,
259,Ole Gunnar Solskjær,Norway,Manchester United,19 December 2018[g],21 November 2021,1068,2018–2021,
292,John Carver,England,Newcastle United,2 January 2015[h],10 June 2015,159,2015,
335,Chris Ramsey,England,Queens Park Rangers,3 February 2015[i],4 November 2015,274,2015,
373,Rubén Sellés,Spain,Southampton,12 February 2023[j],28 May 2023,105,2023,


In [15]:
# Add new 3 columns based on Key section in Wiki

pl_coach = pl_coach.assign(
    is_current = np.where(pl_coach['Name'].str.contains('†'), 1, 0),
    is_caretaker = np.where(pl_coach['Name'].str.contains('‡'), 1, 0),
    is_non_pl_current = np.where(pl_coach['Name'].str.contains('§'), 1, 0)
)

pl_coach

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current
1,George Graham,Scotland,Arsenal,14 May 1986,21 February 1995,3205,1992–1995,[2],0,0,0
2,Stewart Houston‡,Scotland,Arsenal,22 February 1995,8 June 1995,106,1995,[3],0,1,0
3,Bruce Rioch,Scotland,Arsenal,8 June 1995,12 August 1996,431,1995–1996,[2],0,0,0
4,Stewart Houston‡,Scotland,Arsenal,12 August 1996,13 September 1996,32,1996,[3],0,1,0
5,Pat Rice‡,Northern Ireland,Arsenal,13 September 1996,30 September 1996,17,1996,[4],0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
502,Julen Lopetegui,Spain,Wolverhampton Wanderers,14 November 2022,8 August 2023,267,2022–2023,,0,0,0
503,Gary O'Neil,England,Wolverhampton Wanderers,9 August 2023,15 December 2024,494,2023–2024,,0,0,0
504,Vítor Pereira,Portugal,Wolverhampton Wanderers,19 December 2024,2 November 2025,318,2024–2025,,0,0,0
505,James Collins‡,England,Wolverhampton Wanderers,2 November 2025,12 November 2025,10,2025,,0,1,0


In [16]:
# add 2 new columns to check if any invalid date format to set as NaT

pl_coach['from_check'] = pd.to_datetime(pl_coach['From'], errors='coerce').astype(str)
pl_coach['until_check'] = pd.to_datetime(pl_coach['Until'], errors='coerce').astype(str)

In [17]:
# Check if any NaT from those 2 columns (From and Until)

pl_coach[pl_coach['from_check'].str.contains('NaT', case=False) | pl_coach['until_check'].str.contains('NaT', case=False)]

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check
9,Mikel Arteta†,Spain,Arsenal,22 December 2019,Present*,2309,2019–,[8],1,0,0,2019-12-22,NaT
32,Unai Emery†,Spain,Aston Villa,1 November 2022,Present*,1264,2022–,,1,0,0,2022-11-01,NaT
57,Gary O'Neil,England,Bournemouth,30 August 2022[a],19 June 2023,293,2022–2023,[36],0,0,0,NaT,2023-06-19
58,Andoni Iraola†,Spain,Bournemouth,19 June 2023,Present*,1034,2023–,[37],1,0,0,2023-06-19,NaT
64,Keith Andrews†,Republic of Ireland,Brentford,27 June 2025,Present*,295,2025–,,1,0,0,2025-06-27,NaT
67,Roberto De Zerbi,Italy,Brighton & Hove Albion,18 September 2022[b],19 May 2024,1308,2022–2024,,0,0,0,NaT,2024-05-19
68,Fabian Hürzeler†,Germany,Brighton & Hove Albion,2 July 2024,Present*,655,2024–,,1,0,0,2024-07-02,NaT
74,Scott Parker†,England,Burnley,5 July 2024,Present*,652,2025–,,1,0,0,2024-07-05,NaT
113,Liam Rosenior†,England,Chelsea,8 January 2026,Present*,100,2026–,,1,0,0,2026-01-08,NaT
140,Oliver Glasner†,Austria,Crystal Palace,19 February 2024,Present*,789,2024–,,1,0,0,2024-02-19,NaT


In [18]:
# Check current manager with "Present" as the value in "Until" column

pl_coach[pl_coach['Until'].astype(str).str.contains('present', case=False)]

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check
9,Mikel Arteta†,Spain,Arsenal,22 December 2019,Present*,2309,2019–,[8],1,0,0,2019-12-22,NaT
32,Unai Emery†,Spain,Aston Villa,1 November 2022,Present*,1264,2022–,,1,0,0,2022-11-01,NaT
58,Andoni Iraola†,Spain,Bournemouth,19 June 2023,Present*,1034,2023–,[37],1,0,0,2023-06-19,NaT
64,Keith Andrews†,Republic of Ireland,Brentford,27 June 2025,Present*,295,2025–,,1,0,0,2025-06-27,NaT
68,Fabian Hürzeler†,Germany,Brighton & Hove Albion,2 July 2024,Present*,655,2024–,,1,0,0,2024-07-02,NaT
74,Scott Parker†,England,Burnley,5 July 2024,Present*,652,2025–,,1,0,0,2024-07-05,NaT
113,Liam Rosenior†,England,Chelsea,8 January 2026,Present*,100,2026–,,1,0,0,2026-01-08,NaT
140,Oliver Glasner†,Austria,Crystal Palace,19 February 2024,Present*,789,2024–,,1,0,0,2024-02-19,NaT
168,David Moyes†,Scotland,Everton,11 January 2025,Present*,462,2025–,,1,0,0,2025-01-11,NaT
182,Marco Silva†,Portugal,Fulham,1 July 2021,Present*,1752,2022–,,1,0,0,2021-07-01,NaT


In [19]:
# add new column to assign until_at as a valid date column (Until)

pl_coach = pl_coach.assign(
    until_at = np.where(
        pl_coach['Until'].astype(str).str.contains('present', case=False), 
        '31 December 2199', # Max limitation from datetime only until 2262-04-11, so i make it 2199-12-31 instead of 9999-12-31
        pl_coach['Until']
    )
)
pl_coach['until_at'] = pd.to_datetime(pl_coach['until_at'], format='%d %B %Y', errors='coerce')

pl_coach[pl_coach['Until'].astype(str).str.contains('present', case=False)]

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at
9,Mikel Arteta†,Spain,Arsenal,22 December 2019,Present*,2309,2019–,[8],1,0,0,2019-12-22,NaT,2199-12-31
32,Unai Emery†,Spain,Aston Villa,1 November 2022,Present*,1264,2022–,,1,0,0,2022-11-01,NaT,2199-12-31
58,Andoni Iraola†,Spain,Bournemouth,19 June 2023,Present*,1034,2023–,[37],1,0,0,2023-06-19,NaT,2199-12-31
64,Keith Andrews†,Republic of Ireland,Brentford,27 June 2025,Present*,295,2025–,,1,0,0,2025-06-27,NaT,2199-12-31
68,Fabian Hürzeler†,Germany,Brighton & Hove Albion,2 July 2024,Present*,655,2024–,,1,0,0,2024-07-02,NaT,2199-12-31
74,Scott Parker†,England,Burnley,5 July 2024,Present*,652,2025–,,1,0,0,2024-07-05,NaT,2199-12-31
113,Liam Rosenior†,England,Chelsea,8 January 2026,Present*,100,2026–,,1,0,0,2026-01-08,NaT,2199-12-31
140,Oliver Glasner†,Austria,Crystal Palace,19 February 2024,Present*,789,2024–,,1,0,0,2024-02-19,NaT,2199-12-31
168,David Moyes†,Scotland,Everton,11 January 2025,Present*,462,2025–,,1,0,0,2025-01-11,NaT,2199-12-31
182,Marco Silva†,Portugal,Fulham,1 July 2021,Present*,1752,2022–,,1,0,0,2021-07-01,NaT,2199-12-31


In [20]:
# Check to see all of NaT values in "until_check" already being a valid date in "until_at" column

pl_coach[pl_coach['until_check'].str.contains('NaT', case=False)]

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at
9,Mikel Arteta†,Spain,Arsenal,22 December 2019,Present*,2309,2019–,[8],1,0,0,2019-12-22,NaT,2199-12-31
32,Unai Emery†,Spain,Aston Villa,1 November 2022,Present*,1264,2022–,,1,0,0,2022-11-01,NaT,2199-12-31
58,Andoni Iraola†,Spain,Bournemouth,19 June 2023,Present*,1034,2023–,[37],1,0,0,2023-06-19,NaT,2199-12-31
64,Keith Andrews†,Republic of Ireland,Brentford,27 June 2025,Present*,295,2025–,,1,0,0,2025-06-27,NaT,2199-12-31
68,Fabian Hürzeler†,Germany,Brighton & Hove Albion,2 July 2024,Present*,655,2024–,,1,0,0,2024-07-02,NaT,2199-12-31
74,Scott Parker†,England,Burnley,5 July 2024,Present*,652,2025–,,1,0,0,2024-07-05,NaT,2199-12-31
113,Liam Rosenior†,England,Chelsea,8 January 2026,Present*,100,2026–,,1,0,0,2026-01-08,NaT,2199-12-31
140,Oliver Glasner†,Austria,Crystal Palace,19 February 2024,Present*,789,2024–,,1,0,0,2024-02-19,NaT,2199-12-31
168,David Moyes†,Scotland,Everton,11 January 2025,Present*,462,2025–,,1,0,0,2025-01-11,NaT,2199-12-31
182,Marco Silva†,Portugal,Fulham,1 July 2021,Present*,1752,2022–,,1,0,0,2021-07-01,NaT,2199-12-31


In [21]:
# Remove additional char from "From" column

pl_coach['from_check'] = pl_coach['From'].astype(str).str.replace('\[.*\]', '', regex=True)

# Check
pl_coach[pl_coach['From'].str.contains('\[.*\]', '', regex=True)]

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at
57,Gary O'Neil,England,Bournemouth,30 August 2022[a],19 June 2023,293,2022–2023,[36],0,0,0,30 August 2022,2023-06-19,2023-06-19
67,Roberto De Zerbi,Italy,Brighton & Hove Albion,18 September 2022[b],19 May 2024,1308,2022–2024,,0,0,0,18 September 2022,2024-05-19,2024-05-19
181,Scott Parker,England,Fulham,28 February 2019[c],28 June 2021,851,20192020–2021,,0,0,0,28 February 2019,2021-06-28,2021-06-28
187,Iain Dowie,Northern Ireland,Hull City,17 March 2010[d],9 May 2010,53,2010,[92],0,0,0,17 March 2010,2010-05-09,2010-05-09
189,Mike Phelan,England,Hull City,22 July 2016[e],3 January 2017,165,2016–2017,,0,0,0,22 July 2016,2017-01-03,2017-01-03
217,Craig Shakespeare,England,Leicester City,23 February 2017[f],17 October 2017,236,2017,,0,0,0,23 February 2017,2017-10-17,2017-10-17
259,Ole Gunnar Solskjær,Norway,Manchester United,19 December 2018[g],21 November 2021,1068,2018–2021,,0,0,0,19 December 2018,2021-11-21,2021-11-21
292,John Carver,England,Newcastle United,2 January 2015[h],10 June 2015,159,2015,,0,0,0,2 January 2015,2015-06-10,2015-06-10
335,Chris Ramsey,England,Queens Park Rangers,3 February 2015[i],4 November 2015,274,2015,,0,0,0,3 February 2015,2015-11-04,2015-11-04
373,Rubén Sellés,Spain,Southampton,12 February 2023[j],28 May 2023,105,2023,,0,0,0,12 February 2023,2023-05-28,2023-05-28


In [22]:
# add new column to assign from_at as a valid date column (From)

pl_coach = pl_coach.assign(
from_at = np.where(pl_coach['From'].astype(str).str.contains('\[.*\]', regex=True)
         , pd.to_datetime(pl_coach['from_check'], format='%d %B %Y', errors='coerce')
         , pd.to_datetime(pl_coach['from_check'], errors='coerce'))
)

pl_coach

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at,from_at
1,George Graham,Scotland,Arsenal,14 May 1986,21 February 1995,3205,1992–1995,[2],0,0,0,14 May 1986,1995-02-21,1995-02-21,1986-05-14
2,Stewart Houston‡,Scotland,Arsenal,22 February 1995,8 June 1995,106,1995,[3],0,1,0,22 February 1995,1995-06-08,1995-06-08,1995-02-22
3,Bruce Rioch,Scotland,Arsenal,8 June 1995,12 August 1996,431,1995–1996,[2],0,0,0,8 June 1995,1996-08-12,1996-08-12,1995-06-08
4,Stewart Houston‡,Scotland,Arsenal,12 August 1996,13 September 1996,32,1996,[3],0,1,0,12 August 1996,1996-09-13,1996-09-13,1996-08-12
5,Pat Rice‡,Northern Ireland,Arsenal,13 September 1996,30 September 1996,17,1996,[4],0,1,0,13 September 1996,1996-09-30,1996-09-30,1996-09-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502,Julen Lopetegui,Spain,Wolverhampton Wanderers,14 November 2022,8 August 2023,267,2022–2023,,0,0,0,14 November 2022,2023-08-08,2023-08-08,2022-11-14
503,Gary O'Neil,England,Wolverhampton Wanderers,9 August 2023,15 December 2024,494,2023–2024,,0,0,0,9 August 2023,2024-12-15,2024-12-15,2023-08-09
504,Vítor Pereira,Portugal,Wolverhampton Wanderers,19 December 2024,2 November 2025,318,2024–2025,,0,0,0,19 December 2024,2025-11-02,2025-11-02,2024-12-19
505,James Collins‡,England,Wolverhampton Wanderers,2 November 2025,12 November 2025,10,2025,,0,1,0,2 November 2025,2025-11-12,2025-11-12,2025-11-02


In [23]:
# Convert columns as datetime format

pl_coach['from_check'] = pd.to_datetime(pl_coach['from_at'], errors='coerce').astype(str)
pl_coach['until_check'] = pd.to_datetime(pl_coach['until_at'], errors='coerce').astype(str)

In [24]:
# Check if any NaT value

pl_coach[pl_coach['from_check'].str.contains('NaT', case=False) | pl_coach['until_check'].str.contains('NaT', case=False)]

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at,from_at


In [25]:
# Join table coach and team by team_id key in Premier League (league_id = 1)

coach_team = coach_df.merge(team_df[['team_id', 'name', 'league_id']].rename({'name':'club_name'}, axis=1), how='left', on='team_id')
coach_team = coach_team[coach_team['league_id']==1]
coach_team

,coach_id,name,team_id,nationality,club_name,league_id
0,34,NaN,354,NaN,Crystal Palace,1
1,35,NaN,356,NaN,Sheffield United,1
2,36,NaN,389,NaN,Luton Town,1
3,37,NaN,397,NaN,Brighton & Hove Albion,1
4,38,NaN,402,NaN,Brentford,1
5,39,NaN,563,NaN,West Ham United,1
6,40,NaN,1044,NaN,AFC Bournemouth,1
83,31,Gary O'Neil,76,NaN,Wolverhampton Wanderers,1
84,21,Mikel Arteta,57,Spain,Arsenal,1
85,22,Unai Emery,58,Spain,Aston Villa,1


In [26]:
# Merge historical manager table to coach table

coach_pl = coach_team.merge(pl_coach, how='left', left_on='club_name', right_on='Club')
coach_pl

,coach_id,name,team_id,nationality,club_name,league_id,Name,Nat.,Club,From,...,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at,from_at
0,34,NaN,354,NaN,Crystal Palace,1,Steve Coppell,England,Crystal Palace,3 June 1984,...,3270,1992–1993,[67],0.0,0.0,0.0,1984-06-03,1993-05-17,1993-05-17,1984-06-03
1,34,NaN,354,NaN,Crystal Palace,1,Alan Smith,England,Crystal Palace,3 June 1993,...,711,1994–1995,[67],0.0,0.0,0.0,1993-06-03,1995-05-15,1995-05-15,1993-06-03
2,34,NaN,354,NaN,Crystal Palace,1,Steve Coppell,England,Crystal Palace,27 February 1997,...,379,1997–1998,[67],0.0,0.0,0.0,1997-02-27,1998-03-13,1998-03-13,1997-02-27
3,34,NaN,354,NaN,Crystal Palace,1,Attilio Lombardo‡,Italy,Crystal Palace,13 March 1998,...,47,1998,[67],0.0,1.0,0.0,1998-03-13,1998-04-29,1998-04-29,1998-03-13
4,34,NaN,354,NaN,Crystal Palace,1,Tomas Brolin‡,Sweden,Crystal Palace,13 March 1998,...,47,1998,[67],0.0,1.0,0.0,1998-03-13,1998-04-29,1998-04-29,1998-03-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269,33,Nuno Espirito Santo,351,Portugal,Nottingham Forest,1,Steve Cooper,Wales,Nottingham Forest,21 September 2021,...,819,2022–2023,,0.0,0.0,0.0,2021-09-21,2023-12-19,2023-12-19,2021-09-21
270,33,Nuno Espirito Santo,351,Portugal,Nottingham Forest,1,Nuno Espírito Santo,Portugal,Nottingham Forest,20 December 2023,...,629,2023–2025,,0.0,0.0,0.0,2023-12-20,2025-09-09,2025-09-09,2023-12-20
271,33,Nuno Espirito Santo,351,Portugal,Nottingham Forest,1,Ange Postecoglou,Australia,Nottingham Forest,9 September 2025,...,39,2025,,0.0,0.0,0.0,2025-09-09,2025-10-18,2025-10-18,2025-09-09
272,33,Nuno Espirito Santo,351,Portugal,Nottingham Forest,1,Sean Dyche,England,Nottingham Forest,21 October 2025,...,114,2025–2026,[114],0.0,0.0,0.0,2025-10-21,2026-02-12,2026-02-12,2025-10-21


In [27]:
# Check if any null values

coach_pl.isnull().any()

coach_id             False
name                  True
team_id              False
nationality           True
club_name            False
league_id            False
Name                  True
Nat.                  True
Club                  True
From                  True
Until                 True
Duration(days)        True
Years inLeague        True
Ref.                  True
is_current            True
is_caretaker          True
is_non_pl_current     True
from_check            True
until_check           True
until_at              True
from_at               True
dtype: bool

In [28]:
# Check null values based on Name column

coach_pl[coach_pl['Name'].isnull()]

,coach_id,name,team_id,nationality,club_name,league_id,Name,Nat.,Club,From,...,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at,from_at
54,40,NaN,1044,NaN,AFC Bournemouth,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT


In [29]:
# Check club with null values (Bournemouth)

pl_coach[pl_coach['Club'].str.contains('bournemouth', case=False)]

,Name,Nat.,Club,From,Until,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at,from_at
55,Eddie Howe,England,Bournemouth,12 October 2012,1 August 2020,2850,2015–2020,[35],0,0,0,2012-10-12,2020-08-01,2020-08-01,2012-10-12
56,Scott Parker,England,Bournemouth,28 June 2021,30 August 2022,428,2022,,0,0,0,2021-06-28,2022-08-30,2022-08-30,2021-06-28
57,Gary O'Neil,England,Bournemouth,30 August 2022[a],19 June 2023,293,2022–2023,[36],0,0,0,2022-08-30,2023-06-19,2023-06-19,2022-08-30
58,Andoni Iraola†,Spain,Bournemouth,19 June 2023,Present*,1034,2023–,[37],1,0,0,2023-06-19,2199-12-31,2199-12-31,2023-06-19


In [30]:
# Replace club's name align with table historical manager

pl_coach['Club'] = pl_coach['Club'].str.replace('Bournemouth', 'AFC Bournemouth')

In [31]:
# Retry to merge again

coach_pl = coach_team.merge(pl_coach, how='left', left_on='club_name', right_on='Club')
coach_pl

,coach_id,name,team_id,nationality,club_name,league_id,Name,Nat.,Club,From,...,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at,from_at
0,34,NaN,354,NaN,Crystal Palace,1,Steve Coppell,England,Crystal Palace,3 June 1984,...,3270,1992–1993,[67],0,0,0,1984-06-03,1993-05-17,1993-05-17,1984-06-03
1,34,NaN,354,NaN,Crystal Palace,1,Alan Smith,England,Crystal Palace,3 June 1993,...,711,1994–1995,[67],0,0,0,1993-06-03,1995-05-15,1995-05-15,1993-06-03
2,34,NaN,354,NaN,Crystal Palace,1,Steve Coppell,England,Crystal Palace,27 February 1997,...,379,1997–1998,[67],0,0,0,1997-02-27,1998-03-13,1998-03-13,1997-02-27
3,34,NaN,354,NaN,Crystal Palace,1,Attilio Lombardo‡,Italy,Crystal Palace,13 March 1998,...,47,1998,[67],0,1,0,1998-03-13,1998-04-29,1998-04-29,1998-03-13
4,34,NaN,354,NaN,Crystal Palace,1,Tomas Brolin‡,Sweden,Crystal Palace,13 March 1998,...,47,1998,[67],0,1,0,1998-03-13,1998-04-29,1998-04-29,1998-03-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,33,Nuno Espirito Santo,351,Portugal,Nottingham Forest,1,Steve Cooper,Wales,Nottingham Forest,21 September 2021,...,819,2022–2023,,0,0,0,2021-09-21,2023-12-19,2023-12-19,2021-09-21
273,33,Nuno Espirito Santo,351,Portugal,Nottingham Forest,1,Nuno Espírito Santo,Portugal,Nottingham Forest,20 December 2023,...,629,2023–2025,,0,0,0,2023-12-20,2025-09-09,2025-09-09,2023-12-20
274,33,Nuno Espirito Santo,351,Portugal,Nottingham Forest,1,Ange Postecoglou,Australia,Nottingham Forest,9 September 2025,...,39,2025,,0,0,0,2025-09-09,2025-10-18,2025-10-18,2025-09-09
275,33,Nuno Espirito Santo,351,Portugal,Nottingham Forest,1,Sean Dyche,England,Nottingham Forest,21 October 2025,...,114,2025–2026,[114],0,0,0,2025-10-21,2026-02-12,2026-02-12,2025-10-21


In [32]:
# Check if any null value

coach_pl.isnull().any()

coach_id             False
name                  True
team_id              False
nationality           True
club_name            False
league_id            False
Name                 False
Nat.                 False
Club                 False
From                 False
Until                False
Duration(days)       False
Years inLeague       False
Ref.                 False
is_current           False
is_caretaker         False
is_non_pl_current    False
from_check           False
until_check          False
until_at             False
from_at              False
dtype: bool

In [33]:
coach_pl[coach_pl['name'].isnull()].head(5)

,coach_id,name,team_id,nationality,club_name,league_id,Name,Nat.,Club,From,...,Duration(days),Years inLeague,Ref.,is_current,is_caretaker,is_non_pl_current,from_check,until_check,until_at,from_at
0,34,NaN,354,NaN,Crystal Palace,1,Steve Coppell,England,Crystal Palace,3 June 1984,...,3270,1992–1993,[67],0,0,0,1984-06-03,1993-05-17,1993-05-17,1984-06-03
1,34,NaN,354,NaN,Crystal Palace,1,Alan Smith,England,Crystal Palace,3 June 1993,...,711,1994–1995,[67],0,0,0,1993-06-03,1995-05-15,1995-05-15,1993-06-03
2,34,NaN,354,NaN,Crystal Palace,1,Steve Coppell,England,Crystal Palace,27 February 1997,...,379,1997–1998,[67],0,0,0,1997-02-27,1998-03-13,1998-03-13,1997-02-27
3,34,NaN,354,NaN,Crystal Palace,1,Attilio Lombardo‡,Italy,Crystal Palace,13 March 1998,...,47,1998,[67],0,1,0,1998-03-13,1998-04-29,1998-04-29,1998-03-13
4,34,NaN,354,NaN,Crystal Palace,1,Tomas Brolin‡,Sweden,Crystal Palace,13 March 1998,...,47,1998,[67],0,1,0,1998-03-13,1998-04-29,1998-04-29,1998-03-13


In [34]:
coach_pl.columns

Index(['coach_id', 'name', 'team_id', 'nationality', 'club_name', 'league_id',
       'Name', 'Nat.', 'Club', 'From', 'Until', 'Duration(days)',
       'Years inLeague', 'Ref.', 'is_current', 'is_caretaker',
       'is_non_pl_current', 'from_check', 'until_check', 'until_at',
       'from_at'],
      dtype='object')

In [35]:
# Keep the "Name" column instead of "name" as the manager's name

coach_pl_scd = coach_pl[['coach_id', 'team_id', 'club_name', 'league_id','Name', 'Nat.','is_current', 'is_caretaker', 'is_non_pl_current','from_at','until_at']]
coach_pl_scd

,coach_id,team_id,club_name,league_id,Name,Nat.,is_current,is_caretaker,is_non_pl_current,from_at,until_at
0,34,354,Crystal Palace,1,Steve Coppell,England,0,0,0,1984-06-03,1993-05-17
1,34,354,Crystal Palace,1,Alan Smith,England,0,0,0,1993-06-03,1995-05-15
2,34,354,Crystal Palace,1,Steve Coppell,England,0,0,0,1997-02-27,1998-03-13
3,34,354,Crystal Palace,1,Attilio Lombardo‡,Italy,0,1,0,1998-03-13,1998-04-29
4,34,354,Crystal Palace,1,Tomas Brolin‡,Sweden,0,1,0,1998-03-13,1998-04-29
...,...,...,...,...,...,...,...,...,...,...,...
272,33,351,Nottingham Forest,1,Steve Cooper,Wales,0,0,0,2021-09-21,2023-12-19
273,33,351,Nottingham Forest,1,Nuno Espírito Santo,Portugal,0,0,0,2023-12-20,2025-09-09
274,33,351,Nottingham Forest,1,Ange Postecoglou,Australia,0,0,0,2025-09-09,2025-10-18
275,33,351,Nottingham Forest,1,Sean Dyche,England,0,0,0,2025-10-21,2026-02-12


In [36]:
# Check if any null values

coach_pl_scd.isnull().any()

coach_id             False
team_id              False
club_name            False
league_id            False
Name                 False
Nat.                 False
is_current           False
is_caretaker         False
is_non_pl_current    False
from_at              False
until_at             False
dtype: bool

In [37]:
# rename columns

coach_pl_scd = coach_pl_scd.rename({
    'Name':'coach_name',
    'Nat.':'nationality',
    'club_name': 'team_name',
    'from_at': 'valid_from',
    'until_at': 'valid_to'
}, axis=1)

coach_pl_scd

,coach_id,team_id,team_name,league_id,coach_name,nationality,is_current,is_caretaker,is_non_pl_current,valid_from,valid_to
0,34,354,Crystal Palace,1,Steve Coppell,England,0,0,0,1984-06-03,1993-05-17
1,34,354,Crystal Palace,1,Alan Smith,England,0,0,0,1993-06-03,1995-05-15
2,34,354,Crystal Palace,1,Steve Coppell,England,0,0,0,1997-02-27,1998-03-13
3,34,354,Crystal Palace,1,Attilio Lombardo‡,Italy,0,1,0,1998-03-13,1998-04-29
4,34,354,Crystal Palace,1,Tomas Brolin‡,Sweden,0,1,0,1998-03-13,1998-04-29
...,...,...,...,...,...,...,...,...,...,...,...
272,33,351,Nottingham Forest,1,Steve Cooper,Wales,0,0,0,2021-09-21,2023-12-19
273,33,351,Nottingham Forest,1,Nuno Espírito Santo,Portugal,0,0,0,2023-12-20,2025-09-09
274,33,351,Nottingham Forest,1,Ange Postecoglou,Australia,0,0,0,2025-09-09,2025-10-18
275,33,351,Nottingham Forest,1,Sean Dyche,England,0,0,0,2025-10-21,2026-02-12


In [38]:
# remove special char (†|‡|§) in coach_name

coach_pl_scd['coach_name'] = coach_pl_scd['coach_name'].str.replace('†|‡|§', '', regex=True)
coach_pl_scd

,coach_id,team_id,team_name,league_id,coach_name,nationality,is_current,is_caretaker,is_non_pl_current,valid_from,valid_to
0,34,354,Crystal Palace,1,Steve Coppell,England,0,0,0,1984-06-03,1993-05-17
1,34,354,Crystal Palace,1,Alan Smith,England,0,0,0,1993-06-03,1995-05-15
2,34,354,Crystal Palace,1,Steve Coppell,England,0,0,0,1997-02-27,1998-03-13
3,34,354,Crystal Palace,1,Attilio Lombardo,Italy,0,1,0,1998-03-13,1998-04-29
4,34,354,Crystal Palace,1,Tomas Brolin,Sweden,0,1,0,1998-03-13,1998-04-29
...,...,...,...,...,...,...,...,...,...,...,...
272,33,351,Nottingham Forest,1,Steve Cooper,Wales,0,0,0,2021-09-21,2023-12-19
273,33,351,Nottingham Forest,1,Nuno Espírito Santo,Portugal,0,0,0,2023-12-20,2025-09-09
274,33,351,Nottingham Forest,1,Ange Postecoglou,Australia,0,0,0,2025-09-09,2025-10-18
275,33,351,Nottingham Forest,1,Sean Dyche,England,0,0,0,2025-10-21,2026-02-12


In [ ]:
# Add column created_datetime

coach_pl_scd['created_datetime'] = pd.to_datetime(datetime.now())
coach_pl_scd

,coach_id,team_id,team_name,league_id,coach_name,nationality,is_current,is_caretaker,is_non_pl_current,valid_from,valid_to,created_datetime
0,34,354,Crystal Palace,1,Steve Coppell,England,0,0,0,1984-06-03,1993-05-17,2026-04-18 20:00:41.695975
1,34,354,Crystal Palace,1,Alan Smith,England,0,0,0,1993-06-03,1995-05-15,2026-04-18 20:00:41.695975
2,34,354,Crystal Palace,1,Steve Coppell,England,0,0,0,1997-02-27,1998-03-13,2026-04-18 20:00:41.695975
3,34,354,Crystal Palace,1,Attilio Lombardo,Italy,0,1,0,1998-03-13,1998-04-29,2026-04-18 20:00:41.695975
4,34,354,Crystal Palace,1,Tomas Brolin,Sweden,0,1,0,1998-03-13,1998-04-29,2026-04-18 20:00:41.695975
...,...,...,...,...,...,...,...,...,...,...,...,...
272,33,351,Nottingham Forest,1,Steve Cooper,Wales,0,0,0,2021-09-21,2023-12-19,2026-04-18 20:00:41.695975
273,33,351,Nottingham Forest,1,Nuno Espírito Santo,Portugal,0,0,0,2023-12-20,2025-09-09,2026-04-18 20:00:41.695975
274,33,351,Nottingham Forest,1,Ange Postecoglou,Australia,0,0,0,2025-09-09,2025-10-18,2026-04-18 20:00:41.695975
275,33,351,Nottingham Forest,1,Sean Dyche,England,0,0,0,2025-10-21,2026-02-12,2026-04-18 20:00:41.695975


In [39]:
coach_pl_scd.isnull().any()

coach_id             False
team_id              False
team_name            False
league_id            False
coach_name           False
nationality          False
is_current           False
is_caretaker         False
is_non_pl_current    False
valid_from           False
valid_to             False
dtype: bool

In [44]:
# check details columns

coach_pl_scd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   coach_id           277 non-null    int64         
 1   team_id            277 non-null    int64         
 2   team_name          277 non-null    object        
 3   league_id          277 non-null    int64         
 4   coach_name         277 non-null    object        
 5   nationality        277 non-null    object        
 6   is_current         277 non-null    int64         
 7   is_caretaker       277 non-null    int64         
 8   is_non_pl_current  277 non-null    int64         
 9   valid_from         277 non-null    datetime64[ns]
 10  valid_to           277 non-null    datetime64[ns]
 11  created_datetime   277 non-null    datetime64[ns]
dtypes: datetime64[ns](3), int64(6), object(3)
memory usage: 26.1+ KB


In [45]:
# Load the csv file to database

pipe.export_df_to_sql(coach_pl_scd, "coaches_scd")

--- Checking/Creating Database: 'football' ---
Database 'football' is ready.
--- Exporting DataFrame to table 'coaches_scd' ---
Successfully exported 277 rows to 'coaches_scd'.
